## Setup notes

- **Validation data.** `data/test.csv` is the Kaggle test file: it has only `id` and `text`, no labels, so no AUROC can be computed on it (the assignment also says to use *labeled validation data*). Validation here is therefore a **stratified 20% hold-out of `data/train.csv`** (`random_state=42`), the same split the team used in Part 4 (`fangfeidong/part4.py`). The model is trained on the remaining 8,000 rows and never sees the 2,000 validation rows.
- **Features.** 768-d embeddings from `embeddinggemma-2:270m` served by a local Ollama daemon (`ollama serve`), batched as in `lr.ipynb`. Embeddings are cached to `data/train_embeddinggemma2_270m.npy` so re-runs skip the ~6 min embedding step.
- **Model.** `SVC(kernel="poly", C=10.0)`. `probability=True` is added only to obtain P(AI) (Platt scaling) for the error analysis; it does not change the fitted decision boundary.

In [1]:
import os
import re
import time
import warnings

import numpy as np
import pandas as pd
import ollama

from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

# sklearn 1.9 deprecates SVC(probability=True); it still works and matches the team's setup
warnings.filterwarnings("ignore", category=FutureWarning, module="sklearn")

RANDOM_SEED = 42
EMBED_MODEL = "embeddinggemma-2:270m"
THRESHOLD = 0.5  # P(AI) >= 0.5 -> predicted AI

df = pd.read_csv("./data/train.csv")
print(df.shape, df["label"].value_counts().to_dict())
print("test.csv columns:", list(pd.read_csv("./data/test.csv", nrows=1).columns), "(no label column)")

(10000, 3) {0: 5000, 1: 5000}
test.csv columns: ['id', 'text'] (no label column)


## Embed with EmbeddingGemma2:270M

In [2]:
def embed_texts(texts, model=EMBED_MODEL, batch_size=32):
    """Embed texts with Ollama in small batches (same approach as lr.ipynb)."""
    texts = np.asarray(texts, dtype=str).tolist()
    vectors = []
    for start in range(0, len(texts), batch_size):
        response = ollama.embed(model=model, input=texts[start : start + batch_size])
        vectors.extend(response["embeddings"])
    return np.asarray(vectors, dtype=np.float32)


CACHE_PATH = "./data/train_embeddinggemma2_270m.npy"

if os.path.exists(CACHE_PATH):
    X = np.load(CACHE_PATH)
    print(f"Loaded cached embeddings from {CACHE_PATH}")
else:
    start_time = time.time()
    X = embed_texts(df["text"])
    np.save(CACHE_PATH, X)
    print(f"Embedding time: {time.time() - start_time:.1f} seconds")

assert len(X) == len(df)
print(X.shape)

Loaded cached embeddings from ./data/train_embeddinggemma2_270m.npy
(10000, 768)


## Train / validation split and model

In [3]:
y = df["label"].to_numpy()
indices = np.arange(len(df))

train_idx, val_idx = train_test_split(
    indices, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)
X_train, X_val = X[train_idx], X[val_idx]
y_train, y_val = y[train_idx], y[val_idx]
print("train:", X_train.shape, "val:", X_val.shape)

model = SVC(kernel="poly", C=10.0, probability=True, random_state=RANDOM_SEED)

start_time = time.time()
model.fit(X_train, y_train)
print(f"Training time: {time.time() - start_time:.1f} seconds")

train: (8000, 768) val: (2000, 768)
Training time: 33.2 seconds


## Q4.1 — training vs validation AUROC

In [4]:
p_train = model.predict_proba(X_train)[:, 1]
p_val = model.predict_proba(X_val)[:, 1]

train_auroc = roc_auc_score(y_train, p_train)
val_auroc = roc_auc_score(y_val, p_val)

print(f"Train AUROC: {train_auroc:.4f}   accuracy@{THRESHOLD}: {accuracy_score(y_train, p_train >= THRESHOLD):.4f}")
print(f"Val   AUROC: {val_auroc:.4f}   accuracy@{THRESHOLD}: {accuracy_score(y_val, p_val >= THRESHOLD):.4f}")
print(f"AUROC gap (train - val): {train_auroc - val_auroc:.4f}")

# evidence for the gap discussion: how much of the training set the SVM memorises
n_sv = int(model.n_support_.sum())
print(f"Support vectors: {n_sv} of {len(X_train)} training rows ({n_sv / len(X_train):.1%})")
print(f"Features: {X_train.shape[1]} dims, default degree-{model.degree} polynomial kernel")

Train AUROC: 1.0000   accuracy@0.5: 1.0000
Val   AUROC: 0.9722   accuracy@0.5: 0.9200
AUROC gap (train - val): 0.0278
Support vectors: 3875 of 8000 training rows (48.4%)
Features: 768 dims, default degree-3 polynomial kernel


## Q4.2 — validation AUROC by text length

Bins are defined on the whitespace-separated word count of each passage: **short** (< 160 words), **medium** (160–219), **long** (≥ 220). The cut points are roughly the 25th and 55th percentiles, so every bin has ≥ 500 rows and both classes. 95% intervals are from 1,000 bootstrap resamples within each bin.

In [5]:
val = df.iloc[val_idx].copy()
val["p_ai"] = p_val
val["pred"] = (val["p_ai"] >= THRESHOLD).astype(int)
val["n_words"] = val["text"].str.split().str.len()

print(val["n_words"].describe().round(1).to_string())

BINS = [0, 160, 220, np.inf]
BIN_LABELS = ["short (<160 words)", "medium (160-219 words)", "long (>=220 words)"]
val["length_bin"] = pd.cut(val["n_words"], bins=BINS, labels=BIN_LABELS, right=False)


def bootstrap_auroc_ci(labels, scores, n_boot=1000, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    labels, scores = np.asarray(labels), np.asarray(scores)
    stats = []
    for _ in range(n_boot):
        sample = rng.integers(0, len(labels), len(labels))
        if labels[sample].min() == labels[sample].max():
            continue
        stats.append(roc_auc_score(labels[sample], scores[sample]))
    return np.percentile(stats, [2.5, 97.5])


rows = []
for name, group in val.groupby("length_bin", observed=True):
    low, high = bootstrap_auroc_ci(group["label"], group["p_ai"])
    rows.append({
        "bin": name,
        "count": len(group),
        "AI share": round(group["label"].mean(), 3),
        "val AUROC": round(roc_auc_score(group["label"], group["p_ai"]), 4),
        "95% CI": f"[{low:.3f}, {high:.3f}]",
        "accuracy": round(accuracy_score(group["label"], group["pred"]), 3),
        "FP": int(((group["pred"] == 1) & (group["label"] == 0)).sum()),
        "FN": int(((group["pred"] == 0) & (group["label"] == 1)).sum()),
    })
length_table = pd.DataFrame(rows)
length_table

count    2000.0
mean      207.4
std        62.9
min        76.0
25%       159.0
50%       205.0
75%       254.0
max       350.0


,bin,count,AI share,val AUROC,95% CI,accuracy,FP,FN
0,short (<160 words),508,0.579,0.9491,"[0.931, 0.965]",0.894,20,34
1,medium (160-219 words),635,0.400,0.9739,"[0.961, 0.985]",0.915,38,16
2,long (>=220 words),857,0.527,0.9847,"[0.977, 0.990]",0.939,16,36


> Since the embedding model project paragaphs of different length to fixed size vector (768 dim). Shorter paragraph might provide less clue to be encoded, leading to less meaningful feature extracted and thus poorer performance on the same trained model. Under that circumstance, a few keyword might over-dominantly influence the embedding and thus trick the model. We see that less of a issue as length of of text increase, and the model is provided more detail to distinguish.

## Q4.3 — false positives and false negatives

Decision threshold: **P(AI) ≥ 0.5 → predicted AI**. False positive (FP) = human text (label 0) predicted AI; false negative (FN) = AI text (label 1) predicted human. The five most confident errors of each type are listed with a short snippet for inspection.

In [6]:
false_positives = val[(val["pred"] == 1) & (val["label"] == 0)].sort_values("p_ai", ascending=False)
false_negatives = val[(val["pred"] == 0) & (val["label"] == 1)].sort_values("p_ai", ascending=True)

n_human, n_ai = int((val["label"] == 0).sum()), int((val["label"] == 1).sum())
print(f"False positives: {len(false_positives)} of {n_human} human texts ({len(false_positives) / n_human:.1%})")
print(f"False negatives: {len(false_negatives)} of {n_ai} AI texts ({len(false_negatives) / n_ai:.1%})")

pd.set_option("display.max_colwidth", 260)
examples = pd.concat([
    false_positives.head(5).assign(error="FP"),
    false_negatives.head(5).assign(error="FN"),
])
examples["snippet"] = examples["text"].str.replace(r"\s+", " ", regex=True).str.slice(0, 240)
examples[["id", "error", "p_ai", "n_words", "snippet"]].round({"p_ai": 3})

False positives: 74 of 1000 human texts (7.4%)
False negatives: 86 of 1000 AI texts (8.6%)


,id,error,p_ai,n_words,snippet
688,689,FP,0.993,169,Hey caption is the most basic question ever and more than likely had been asked before but my question is much more different well i believe. I’ve taken on graphic design as an entrepreneurship project where we learn something that can help
3659,3660,FP,0.990,207,"Hector MacLean may refer to: Clan Chiefs Many chiefs of Clan Maclean have been called Hector or Eachann, including: Red Hector of the Battles Maclean (1368–1411), 6th Clan Chief, killed at the Battle of Harlaw, also known as ""Red Hector"" He"
9611,9612,FP,0.984,155,"The Project-Level Aid (PLAID) database has now become AidData: Tracking Development Finance, a portal for information on development finance. As AidData, it now contains records of over 900,000 international development projects financed by"
3742,3743,FP,0.974,153,"my WIP I'm currently working on is a zombie fiction book that takes quite a bit from 28 Days Later in that it focuses on a few select characters and their struggles, fears, and reactions to the world breaking down around them, and how their"
8476,8477,FP,0.970,159,"I made a Hefeweizen with mostly Pilsner malt and some white wheat. eBIAB five gallon batch if you’re interested. There was some Munich malt for some flavor as well. I wanted to try something different than a traditional Hefeweizen, so I boi"
8485,8486,FN,0.008,84,I am renewing my first class medical and the AME asked me a few questions that I was not expecting. He asked about my history of anxiety and depression and if I had ever been treated for either. He also asked if I had ever been diagnosed wi
4446,4447,FN,0.011,137,"Incorporated in 1768 as Rye Beach and renamed to its present name after a town meeting on June 10th 1803.[1] The first settlers arrived around 1623[2], but it was not until 1643 that they were granted land by Massachusetts Bay Colony Govern"
1361,1362,FN,0.012,233,Jacques Boyceau (1607 â€“ 1642) was a French organist and composer who became a monk of the Congregation of Saint-Maur in 1631. He is best known for his compositions for organ and for theorbo and several of his works were published during h
7836,7837,FN,0.013,108,"When the movie starts, at first you just can think ""Oh, another Zombie movie"" but after that, right away we see 4 people going to California and along with the movie, you know why. As time goes by, we know better every thing, the movie beca"
1040,1041,FN,0.014,194,"Banjarmasin, a city on the island Borneo in Indonesia, has traditionally been located at the intersection of major routes between South and Southeast Asia with China. The Sultan Bung Hatta Museum houses Islamic artifacts that illustrate thi"


## Q4.4 — recurring failure patterns

To check whether the patterns seen in the examples above recur, the cell below tags every validation passage with a few crude, observable surface cues and compares error rates with and without each cue. FP rate is computed over human texts only, FN rate over AI texts only.

In [7]:
text = val["text"]
first_person = text.str.count(r"\b(?:I|I'm|I've|I'd|I’m|I’ve|I’d|my|My)\b") / val["n_words"]

cues = {
    "first-person / forum style (>=3% I/my tokens)": first_person >= 0.03,
    "encyclopedic style (<0.5% I/my tokens)": first_person < 0.005,
    "noisy surface: mojibake, [1]-style citation debris, or lowercase ' i '": (
        text.str.contains(r"â€|Ã.|\[\d+\]|citation needed|\si\s", regex=True)
    ),
    "short (<160 words)": val["n_words"] < 160,
}

is_human, is_ai = val["label"] == 0, val["label"] == 1
is_fp, is_fn = is_human & (val["pred"] == 1), is_ai & (val["pred"] == 0)

rows = []
for name, mask in cues.items():
    for flag, m in (("yes", mask), ("no", ~mask)):
        rows.append({
            "cue": name,
            "has cue": flag,
            "n human": int((m & is_human).sum()),
            "FP rate": round((m & is_fp).sum() / max((m & is_human).sum(), 1), 3),
            "n AI": int((m & is_ai).sum()),
            "FN rate": round((m & is_fn).sum() / max((m & is_ai).sum(), 1), 3),
        })
cue_table = pd.DataFrame(rows)
cue_table

,cue,has cue,n human,FP rate,n AI,FN rate
0,first-person / forum style (>=3% I/my tokens),yes,179,0.140,121,0.116
1,first-person / forum style (>=3% I/my tokens),no,821,0.060,879,0.082
2,encyclopedic style (<0.5% I/my tokens),yes,713,0.056,744,0.078
3,encyclopedic style (<0.5% I/my tokens),no,287,0.118,256,0.109
4,"noisy surface: mojibake, [1]-style citation debris, or lowercase ' i '",yes,38,0.079,89,0.303
5,"noisy surface: mojibake, [1]-style citation debris, or lowercase ' i '",no,962,0.074,911,0.065
6,short (<160 words),yes,214,0.093,294,0.116
7,short (<160 words),no,786,0.069,706,0.074


In [8]:
# how confident are the errors?
print("FP with P(AI) >= 0.9:", int((false_positives["p_ai"] >= 0.9).sum()), "of", len(false_positives))
print("FN with P(AI) <= 0.1:", int((false_negatives["p_ai"] <= 0.1).sum()), "of", len(false_negatives))
print("Median words  FP:", false_positives["n_words"].median(), " FN:", false_negatives["n_words"].median(),
      " all val:", val["n_words"].median())

FP with P(AI) >= 0.9: 14 of 74
FN with P(AI) <= 0.1: 23 of 86
Median words  FP: 181.0  FN: 192.5  all val: 205.0
